# Combine and Prepare Data

This notebook combines the previously processed ENTSO-E market data and outage data into a unified modelling dataset.

The different datasets are aligned on a common quarter-hourly time index, cleaned, transformed, and extended with additional features required for the modelling workflow.

The code is structured in the following sections:

- **Imports and Settings**
- **Load Fundamental Data**
- **Load and merge Target Data**
- **Preprocessing and Feature Calculation**
- **Export**

In [ ]:
# =========================================================
# IMPORTS AND SETTINGS
# =========================================================

import pandas as pd
import numpy as np
import holidays as hol

time_resolutions = ['qh', 'h']
targets = ['id1', 'id3']

data_frames = {}

In [ ]:
# =========================================================
# LOAD FUNDAMENTAL DATA
# =========================================================

# read csv from "entsoe_data_{time_resolution}.csv" for each time resolution from ../data
for time_resolution in time_resolutions:
    df_api_data = pd.read_csv(f"../data/entsoe_data_{time_resolution}.csv", parse_dates=['timestamp_local'])
    df_api_data["timestamp_local"] = pd.to_datetime(df_api_data["timestamp_local"], utc=True).dt.tz_convert("Europe/Vienna")
    df_api_data = df_api_data.rename(columns={"timestamp_local": "datetime"})
    data_frames[time_resolution] = df_api_data

# drop columns with only NaN values
for time_resolution, df in data_frames.items():
    data_frames[time_resolution] = df.dropna(axis=1, how='all')

# read and merge outage data
for time_resolution in time_resolutions:
    df_outage_data = pd.read_csv(f"../data/outage_data_{time_resolution}.csv", parse_dates=['datetime'])
    df_outage_data["datetime"] = pd.to_datetime(df_outage_data["datetime"], utc=True).dt.tz_convert("Europe/Vienna")
    data_frames[time_resolution] = data_frames[time_resolution].merge(df_outage_data, on="datetime", how="left")

In [ ]:
# =========================================================
# LOAD AND MERGE TARGET DATA
# =========================================================

# read id1 and id3 prices from Excel for both time resolutions
for time_res in time_resolutions:

    file_path = "../data/prices/id1_id3.xlsx"
    sheet_name = time_res
    tz_name = "Europe/Vienna"

    # Columns to read (A, B, C)
    usecols = "A:C"
    skiprows = 5

    # Manual column names
    col_names = ["datetime", "id1", "id3"]

    # ---------------------------------------------------------
    # READ EXCEL
    # ----------------------------------------------------------

    df_prices = pd.read_excel(
        file_path,
        sheet_name=sheet_name,
        usecols=usecols,
        skiprows=skiprows,
        header=None,
    )

    df_prices.columns = col_names

    # ----------------------------------------------------------
    # PARSE DATETIME
    # ----------------------------------------------------------

    # Parse naive Excel timestamps
    df_prices["datetime"] = pd.to_datetime(df_prices["datetime"])

    # Correct DST handling:
    # first duplicated 02:xx block -> +02:00
    # second duplicated 02:xx block -> +01:00
    df_prices["datetime"] = (
        df_prices["datetime"]
        .dt.tz_localize(
            tz_name,
            ambiguous="infer",
            nonexistent="shift_forward",
        )
    )

    # ----------------------------------------------------------
    # CONVERT PRICES
    # ----------------------------------------------------------

    df_prices["id1"] = pd.to_numeric(
        df_prices["id1"],
        errors="coerce",
    )

    df_prices["id3"] = pd.to_numeric(
        df_prices["id3"],
        errors="coerce",
    )

    print(
        f"✅ Read {len(df_prices)} rows "
        f"from {file_path} (sheet: {sheet_name})"
    )

    # ----------------------------------------------------------
    # MERGE IN UTC
    # ----------------------------------------------------------

    df_prices["datetime"] = pd.to_datetime(
        df_prices["datetime"],
        utc=True,
    )

    df_api_data = data_frames[time_res].copy()

    df_api_data["datetime"] = pd.to_datetime(
        df_api_data["datetime"],
        utc=True,
    )

    df_merged = pd.merge(
        df_api_data,
        df_prices,
        on="datetime",
        how="inner",
    )

    # ----------------------------------------------------------
    # CONVERT BACK TO VIENNA TIME
    # ----------------------------------------------------------

    df_merged["datetime"] = (
        df_merged["datetime"]
        .dt.tz_convert(tz_name)
    )

    # store back in data_frames
    data_frames[time_res] = df_merged

✅ Read 140256 rows from ../data/prices/id1_id3.xlsx (sheet: qh)
✅ Read 70128 rows from ../data/prices/id1_id3.xlsx (sheet: h)


In [ ]:
# =========================================================
# CALCULATE FEATURES AND PREPROCESSING
# =========================================================

# calc time based features
for time_res in time_resolutions:
    # calculate hour of the day and month of the year and day of the week
    data_frames[time_res]["hour"] = data_frames[time_res]["datetime"].dt.hour
    data_frames[time_res]["month"] = data_frames[time_res]["datetime"].dt.month
    data_frames[time_res]["day_of_week"] = data_frames[time_res]["datetime"].dt.weekday

    # create sin and cos features for hour and month
    data_frames[time_res]["hour_sin"] = np.sin(2 * np.pi * data_frames[time_res]["hour"] / 24)
    data_frames[time_res]["hour_cos"] = np.cos(2 * np.pi * data_frames[time_res]["hour"] / 24)
    data_frames[time_res]["month_sin"] = np.sin(2 * np.pi * (data_frames[time_res]["month"] - 1) / 12)
    data_frames[time_res]["month_cos"] = np.cos(2 * np.pi * (data_frames[time_res]["month"] - 1) / 12)

    # create holiday feature
    at_holidays = hol.Austria(years=[2019, 2020, 2021, 2022, 2023, 2024, 2025])
    holidays = set(at_holidays.keys())

    # calculate if a day is a weekend or holiday
    data_frames[time_res]["is_weekend"] = data_frames[time_res]["datetime"].dt.weekday >= 5
    data_frames[time_res]["is_holiday"] = data_frames[time_res]["datetime"].dt.date.isin(holidays)
    data_frames[time_res]["free_day"] = (data_frames[time_res]["is_weekend"] | data_frames[time_res]["is_holiday"]).astype(int)

In [10]:
# calc import features day-ahead and total

for time_res in time_resolutions:
    # day-ahead imports
    # calc schduled imports for day-ahead (positive means import, negative means export)
    data_frames[time_res]["net_import_day_ahead"] = data_frames[time_res]["DE_to_AT_day_ahead"] - data_frames[time_res]["AT_to_DE_day_ahead"] + \
        data_frames[time_res]["CH_to_AT_day_ahead"] - data_frames[time_res]["AT_to_CH_day_ahead"] + \
        data_frames[time_res]["CZ_to_AT_day_ahead"] - data_frames[time_res]["AT_to_CZ_day_ahead"] + \
        data_frames[time_res]["IT_to_AT_day_ahead"] - data_frames[time_res]["AT_to_IT_day_ahead"] + \
        data_frames[time_res]["HU_to_AT_day_ahead"] - data_frames[time_res]["AT_to_HU_day_ahead"] + \
        data_frames[time_res]["SI_to_AT_day_ahead"] - data_frames[time_res]["AT_to_SI_day_ahead"]
    # total imports
    data_frames[time_res]["net_import_total"] = data_frames[time_res]["DE_to_AT_total"] - data_frames[time_res]["AT_to_DE_total"] + \
        data_frames[time_res]["CH_to_AT_total"] - data_frames[time_res]["AT_to_CH_total"] + \
        data_frames[time_res]["CZ_to_AT_total"] - data_frames[time_res]["AT_to_CZ_total"] + \
        data_frames[time_res]["IT_to_AT_total"] - data_frames[time_res]["AT_to_IT_total"] + \
        data_frames[time_res]["HU_to_AT_total"] - data_frames[time_res]["AT_to_HU_total"] + \
        data_frames[time_res]["SI_to_AT_total"] - data_frames[time_res]["AT_to_SI_total"]
    
    # only import day-ahead
    data_frames[time_res]["import_day_ahead"] = data_frames[time_res]["DE_to_AT_day_ahead"] + \
        data_frames[time_res]["CH_to_AT_day_ahead"] + \
        data_frames[time_res]["CZ_to_AT_day_ahead"] + \
        data_frames[time_res]["IT_to_AT_day_ahead"] + \
        data_frames[time_res]["HU_to_AT_day_ahead"] + \
        data_frames[time_res]["SI_to_AT_day_ahead"]
    
    # only export day-ahead
    data_frames[time_res]["export_day_ahead"] = data_frames[time_res]["AT_to_DE_day_ahead"] + \
        data_frames[time_res]["AT_to_CH_day_ahead"] + \
        data_frames[time_res]["AT_to_CZ_day_ahead"] + \
        data_frames[time_res]["AT_to_IT_day_ahead"] + \
        data_frames[time_res]["AT_to_HU_day_ahead"] + \
        data_frames[time_res]["AT_to_SI_day_ahead"]
    
    # only import total
    data_frames[time_res]["import_total"] = data_frames[time_res]["DE_to_AT_total"] + \
        data_frames[time_res]["CH_to_AT_total"] + \
        data_frames[time_res]["CZ_to_AT_total"] + \
        data_frames[time_res]["IT_to_AT_total"] + \
        data_frames[time_res]["HU_to_AT_total"] + \
        data_frames[time_res]["SI_to_AT_total"]
    
    # only export total
    data_frames[time_res]["export_total"] = data_frames[time_res]["AT_to_DE_total"] + \
        data_frames[time_res]["AT_to_CH_total"] + \
        data_frames[time_res]["AT_to_CZ_total"] + \
        data_frames[time_res]["AT_to_IT_total"] + \
        data_frames[time_res]["AT_to_HU_total"] + \
        data_frames[time_res]["AT_to_SI_total"]
    
    # delta of total vs day-ahead for imports/eports/net_imports
    data_frames[time_res]["import_delta"] = data_frames[time_res]["import_total"] - data_frames[time_res]["import_day_ahead"]
    data_frames[time_res]["export_delta"] = data_frames[time_res]["export_total"] - data_frames[time_res]["export_day_ahead"]
    data_frames[time_res]["net_import_delta"] = data_frames[time_res]["net_import_total"] - data_frames[time_res]["net_import_day_ahead"]
    
    # calc delta between day-ahead and total imports
    data_frames[time_res]["DE_to_AT_delta"] = data_frames[time_res]["DE_to_AT_total"] - data_frames[time_res]["DE_to_AT_day_ahead"]
    data_frames[time_res]["CH_to_AT_delta"] = data_frames[time_res]["CH_to_AT_total"] - data_frames[time_res]["CH_to_AT_day_ahead"]
    data_frames[time_res]["CZ_to_AT_delta"] = data_frames[time_res]["CZ_to_AT_total"] - data_frames[time_res]["CZ_to_AT_day_ahead"]
    data_frames[time_res]["IT_to_AT_delta"] = data_frames[time_res]["IT_to_AT_total"] - data_frames[time_res]["IT_to_AT_day_ahead"]
    data_frames[time_res]["HU_to_AT_delta"] = data_frames[time_res]["HU_to_AT_total"] - data_frames[time_res]["HU_to_AT_day_ahead"]
    data_frames[time_res]["SI_to_AT_delta"] = data_frames[time_res]["SI_to_AT_total"] - data_frames[time_res]["SI_to_AT_day_ahead"]
    data_frames[time_res]["AT_to_DE_delta"] = data_frames[time_res]["AT_to_DE_total"] - data_frames[time_res]["AT_to_DE_day_ahead"]
    data_frames[time_res]["AT_to_CH_delta"] = data_frames[time_res]["AT_to_CH_total"] - data_frames[time_res]["AT_to_CH_day_ahead"]
    data_frames[time_res]["AT_to_CZ_delta"] = data_frames[time_res]["AT_to_CZ_total"] - data_frames[time_res]["AT_to_CZ_day_ahead"]
    data_frames[time_res]["AT_to_IT_delta"] = data_frames[time_res]["AT_to_IT_total"] - data_frames[time_res]["AT_to_IT_day_ahead"]
    data_frames[time_res]["AT_to_HU_delta"] = data_frames[time_res]["AT_to_HU_total"] - data_frames[time_res]["AT_to_HU_day_ahead"]
    data_frames[time_res]["AT_to_SI_delta"] = data_frames[time_res]["AT_to_SI_total"] - data_frames[time_res]["AT_to_SI_day_ahead"]

    # calc net import delta per country
    data_frames[time_res]["DE_net_import_delta"] = data_frames[time_res]["DE_to_AT_delta"] - data_frames[time_res]["AT_to_DE_delta"]
    data_frames[time_res]["CH_net_import_delta"] = data_frames[time_res]["CH_to_AT_delta"] - data_frames[time_res]["AT_to_CH_delta"]
    data_frames[time_res]["CZ_net_import_delta"] = data_frames[time_res]["CZ_to_AT_delta"] - data_frames[time_res]["AT_to_CZ_delta"]
    data_frames[time_res]["IT_net_import_delta"] = data_frames[time_res]["IT_to_AT_delta"] - data_frames[time_res]["AT_to_IT_delta"]
    data_frames[time_res]["HU_net_import_delta"] = data_frames[time_res]["HU_to_AT_delta"] - data_frames[time_res]["AT_to_HU_delta"]
    data_frames[time_res]["SI_net_import_delta"] = data_frames[time_res]["SI_to_AT_delta"] - data_frames[time_res]["AT_to_SI_delta"]

    # calc total "intraday flow"
    data_frames[time_res]["total_intraday_flow"] = data_frames[time_res]["DE_to_AT_delta"] + data_frames[time_res]["CH_to_AT_delta"] + \
        data_frames[time_res]["CZ_to_AT_delta"] + data_frames[time_res]["IT_to_AT_delta"] + \
        data_frames[time_res]["HU_to_AT_delta"] + data_frames[time_res]["SI_to_AT_delta"] + data_frames[time_res]["AT_to_DE_delta"] + \
        data_frames[time_res]["AT_to_CH_delta"] + data_frames[time_res]["AT_to_CZ_delta"] + data_frames[time_res]["AT_to_IT_delta"] + \
        data_frames[time_res]["AT_to_HU_delta"] + data_frames[time_res]["AT_to_SI_delta"]

In [ ]:
# calc deltas for target variables (id1 and id3)
for time_res in time_resolutions: 
    data_frames[time_res]["id1_delta"] = data_frames[time_res]["id1"] - data_frames[time_res]["day_ahead_price"]
    data_frames[time_res]["id3_delta"] = data_frames[time_res]["id3"] - data_frames[time_res]["day_ahead_price"]

    # calc load delta
    data_frames[time_res]["load_delta"] = data_frames[time_res]["load_actual_mw"] - data_frames[time_res]["load_forecast_mw"]

    # calc renewable deltas
    data_frames[time_res]["wind_delta"] = data_frames[time_res]["wind_onshore_mw"] - data_frames[time_res]["wind_day_ahead"] 
    data_frames[time_res]["solar_delta"] = data_frames[time_res]["solar_mw"] - data_frames[time_res]["solar_day_ahead"]

    # calc detla current-day-ahead for wind/solar
    data_frames[time_res]["wind_delta_c"] = data_frames[time_res]["wind_current"] - data_frames[time_res]["wind_day_ahead"]
    data_frames[time_res]["solar_delta_c"] = data_frames[time_res]["solar_current"] - data_frames[time_res]["solar_day_ahead"]

    # calc residual load = actual load - (wind + solar + hydro run-of-river)
    data_frames[time_res]["residual_load"] = data_frames[time_res]["load_actual_mw"] \
        - data_frames[time_res]["wind_onshore_mw"] - data_frames[time_res]["solar_mw"] - data_frames[time_res]["hydro_run_of_river_and_poundage_mw"]
    
    # total production
    data_frames[time_res]["total_production_mw"] = data_frames[time_res]["fossil_gas_mw"] + data_frames[time_res]["biomass_mw"] + \
        data_frames[time_res]["hydro_run_of_river_and_poundage_mw"] + data_frames[time_res]["hydro_water_reservoir_mw"] + \
        data_frames[time_res]["hydro_pumped_storage_mw"] + data_frames[time_res]["solar_mw"] + data_frames[time_res]["wind_onshore_mw"]

    # calc share of gas in total production
    data_frames[time_res]["gas_share"] = data_frames[time_res]["fossil_gas_mw"] / data_frames[time_res]["total_production_mw"]

    # ramp outage (current - previous)
    data_frames[time_res]["ramp_outage"] = data_frames[time_res]["outage_total_true"] - data_frames[time_res]["outage_total_true"].shift(1)

In [ ]:
# =========================================================
# EXPORT
# =========================================================

# export final merged dataframes to csv
for time_res in time_resolutions:
    data_frames[time_res] = data_frames[time_res].set_index("datetime").sort_index()
    data_frames[time_res].to_csv(f"../data/combined_data_{time_res}.csv", index=True)